# Tool Calling in LangChain
### Define tools, bind them to models, and run an agent loop

---
**Topics Covered:**
- Defining tools with the `@tool` decorator
- Binding tools to an LLM with `.bind_tools()`
- Detecting and dispatching tool calls from model responses
- Building a minimal ReAct-style tool-use loop
- Using `ToolMessage` to feed results back to the model


In [2]:
import os, json, math
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings("ignore")

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0)

## 1. Defining Tools with `@tool`

The `@tool` decorator turns a Python function into a LangChain tool.  
The **docstring** becomes the tool description the model reads to decide when to call it.

In [3]:
@tool
def compute_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate Body Mass Index (BMI) given weight in kilograms and height in metres."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI = {bmi:.2f} ({category})"


@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a numeric value between common units.
    Supported conversions: km↔miles, kg↔lbs, celsius↔fahrenheit.
    """
    key = (from_unit.lower(), to_unit.lower())
    conversions = {
        ("km", "miles"):       lambda v: v * 0.621371,
        ("miles", "km"):       lambda v: v * 1.60934,
        ("kg", "lbs"):         lambda v: v * 2.20462,
        ("lbs", "kg"):         lambda v: v * 0.453592,
        ("celsius", "fahrenheit"): lambda v: (v * 9 / 5) + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
    }
    if key not in conversions:
        return f"Conversion from {from_unit} to {to_unit} is not supported."
    result = conversions[key](value)
    return f"{value} {from_unit} = {result:.4f} {to_unit}"


@tool
def text_stats(text: str) -> str:
    """Return basic statistics for a piece of text: word count, character count, sentence count, and average word length."""
    words     = text.split()
    sentences = text.count(".") + text.count("!") + text.count("?")
    avg_len   = sum(len(w) for w in words) / max(len(words), 1)
    return (
        f"Words: {len(words)} | "
        f"Characters: {len(text)} | "
        f"Sentences: {sentences} | "
        f"Avg word length: {avg_len:.1f}"
    )


print("Tools defined:", [compute_bmi.name, unit_converter.name, text_stats.name])

Tools defined: ['compute_bmi', 'unit_converter', 'text_stats']


## 2. Inspect a Tool's Schema

Every `@tool` automatically generates a JSON schema the LLM uses to understand inputs.

In [4]:
import json
print(json.dumps(compute_bmi.args_schema.model_json_schema(), indent=2))

{
  "description": "Calculate Body Mass Index (BMI) given weight in kilograms and height in metres.",
  "properties": {
    "weight_kg": {
      "title": "Weight Kg",
      "type": "number"
    },
    "height_m": {
      "title": "Height M",
      "type": "number"
    }
  },
  "required": [
    "weight_kg",
    "height_m"
  ],
  "title": "compute_bmi",
  "type": "object"
}


## 3. Bind Tools to the LLM

`.bind_tools()` tells the model which tools are available for the current call.

In [5]:
tools = [compute_bmi, unit_converter, text_stats]

# Create a version of the LLM that knows about our tools
llm_with_tools = llm.bind_tools(tools)

# Tool map for fast dispatch
tool_map = {t.name: t for t in tools}
print("Registered tools:", list(tool_map.keys()))

Registered tools: ['compute_bmi', 'unit_converter', 'text_stats']


## 4. Single Tool Call — Model Picks the Right Tool

In [6]:
response = llm_with_tools.invoke("What is the BMI of someone who weighs 72 kg and is 1.75 m tall?")

print("Finish reason  :", response.response_metadata.get("finish_reason"))
print("Tool calls made:", len(response.tool_calls))

if response.tool_calls:
    tc = response.tool_calls[0]
    print("Tool selected  :", tc["name"])
    print("Arguments      :", tc["args"])

Finish reason  : tool_calls
Tool calls made: 1
Tool selected  : compute_bmi
Arguments      : {'height_m': 1.75, 'weight_kg': 72}


## 5. Execute the Tool Call and Return Result to the Model

In [7]:
def execute_tool_call(tc):
    """Run a single tool call dict returned by the LLM."""
    func = tool_map[tc["name"]]
    result = func.invoke(tc["args"])
    return ToolMessage(content=str(result), tool_call_id=tc["id"])


# Run each tool the model requested
tool_results = [execute_tool_call(tc) for tc in response.tool_calls]
for tr in tool_results:
    print("Tool result:", tr.content)

Tool result: BMI = 23.51 (Normal weight)


In [8]:
# Send tool results back to the model for a final answer
messages = [
    HumanMessage(content="What is the BMI of someone who weighs 72 kg and is 1.75 m tall?"),
    response,          # assistant message containing the tool_calls
] + tool_results       # one ToolMessage per call

final = llm_with_tools.invoke(messages)
print(final.content)

The BMI of someone who weighs 72 kg and is 1.75 m tall is 23.51, which falls into the normal weight category.


## 6. Full Agentic Loop Function

Wrap the pattern above into a reusable function that keeps calling tools until `finish_reason == "stop"`.

In [9]:
def run_agent(user_query: str, system_prompt: str = "You are a helpful assistant with access to tools.") -> str:
    """Minimal ReAct loop: think → act (tool) → observe → repeat until done."""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_query),
    ]

    for step in range(6):   # safety cap at 6 iterations
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content   # model is done

        # Execute every tool the model requested this turn
        for tc in response.tool_calls:
            tool_msg = execute_tool_call(tc)
            messages.append(tool_msg)
            print(f"  [{tc['name']}] → {tool_msg.content}")

    return "Agent reached iteration limit without a final answer."

In [10]:
# Test 1: unit conversion
print("=== Unit conversion ===")
print(run_agent("How many miles is 42.195 km (a full marathon)?"))

=== Unit conversion ===
  [unit_converter] → 42.195 km = 26.2187 miles
A full marathon is approximately 26.22 miles.


In [11]:
# Test 2: multiple tools in one query
print("=== Multi-tool ===")
answer = run_agent(
    "A runner weighs 65 kg (how many lbs is that?) and is 1.80 m tall — what is their BMI?"
)
print(answer)

=== Multi-tool ===
  [unit_converter] → 65.0 kg = 143.3003 lbs
  [compute_bmi] → BMI = 20.06 (Normal weight)
The runner weighs 143.3 lbs and has a BMI of 20.06, which falls into the normal weight category.


In [12]:
# Test 3: text stats tool
print("=== Text stats ===")
sample_text = (
    "Large language models are transforming software engineering. "
    "They can generate, summarise, and explain code faster than ever. "
    "Understanding their limitations is equally important!"
)
print(run_agent(f"Analyse this text for me: \"{sample_text}\""))

=== Text stats ===
  [text_stats] → Words: 23 | Characters: 179 | Sentences: 3 | Avg word length: 6.8
The text has 23 words, 179 characters, and 3 sentences. The average word length is 6.8 characters.


## 7. No Tool Needed — Model Answers Directly

In [13]:
print(run_agent("What is the difference between supervised and unsupervised learning?"))

Supervised and unsupervised learning are two fundamental concepts in machine learning, a subset of artificial intelligence that involves training algorithms to make predictions or take actions based on data.

Supervised learning involves training a model on labeled data, where the correct output is already known. The goal is to learn a mapping between input data and the corresponding output labels, so the model can make predictions on new, unseen data. In supervised learning, the algorithm is essentially learning from a teacher who provides the correct answers. Examples of supervised learning tasks include image classification, sentiment analysis, and regression problems.

On the other hand, unsupervised learning involves training a model on unlabeled data, where the goal is to discover patterns, relationships, or groupings within the data. There is no correct output or teacher to guide the learning process. Unsupervised learning algorithms aim to identify structure in the data, such a

---
### Summary

| Step | Code |
|------|------|
| Define tool | `@tool` + docstring |
| Bind tools | `llm.bind_tools([...])` |
| Detect tool call | `response.tool_calls` |
| Execute | `tool.invoke(tc["args"])` |
| Return result | `ToolMessage(content=..., tool_call_id=...)` |

**Next → `04_lcel_chains.ipynb`**